# === Imports ===

### === Core ===

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import json, time, warnings, re
warnings.filterwarnings("ignore")

### === PDF parsing ===

In [28]:
import fitz   

### === Embeddings + vector store ===

In [29]:
from sentence_transformers import SentenceTransformer
import chromadb


### === Local LLM (Ollama) ===

In [30]:
import ollama

### === Progress ===

In [2]:
from tqdm import tqdm
print("RAG notebook ready.")

RAG notebook ready.


# === Paths + constants ===

### === Project root ===

In [3]:
PROJECT_ROOT = Path.cwd()
DATASET_DIR  = PROJECT_ROOT / "Dataset"
OUTPUT_DIR   = PROJECT_ROOT / "outputs"
CHROMA_DIR   = PROJECT_ROOT / "chroma_db"      # persistent vector store
CHROMA_DIR.mkdir(exist_ok=True)

### === Corpus paths ===

In [33]:
TIER1_DIR    = DATASET_DIR / "RAG corpus (medical literatureguidelines)"
TIER2_DIR    = DATASET_DIR / "Tier2_literature"

### === Config ===

In [34]:
EMBED_MODEL  = "all-MiniLM-L6-v2"
LLM_MODEL    = "qwen3.5:9b"
JUDGE_MODEL  = "llama3.1:8b"   # different family from generator -> avoids self-preference bias; fits 8GB VRAM
CHUNK_WORDS  = 500
OVERLAP_WORDS = 50
TOP_K        = 5

### === Handoff from Notebook 1 ===

In [35]:
FLAGGED_PATH = OUTPUT_DIR / "flagged_windows.parquet"
print("Paths set.")
print("  Tier-1:", TIER1_DIR)
print("  Tier-2:", TIER2_DIR)
print("  ChromaDB:", CHROMA_DIR)
print("  LLM:", LLM_MODEL)

Paths set.
  Tier-1: d:\All_Folder\data project\new_papers\Retrieval-augmented generation for continuous anomaly alerts\Dataset\RAG corpus (medical literatureguidelines)
  Tier-2: d:\All_Folder\data project\new_papers\Retrieval-augmented generation for continuous anomaly alerts\Dataset\Tier2_literature
  ChromaDB: d:\All_Folder\data project\new_papers\Retrieval-augmented generation for continuous anomaly alerts\chroma_db
  LLM: qwen3.5:9b


### === Verify the paths ===

In [36]:
print("Tier-1 PDFs:", list(TIER1_DIR.glob("*.pdf")))
print("Tier-2 buckets:", [p.name for p in TIER2_DIR.iterdir() if p.is_dir()])
print("Flagged windows exists:", FLAGGED_PATH.exists())

Tier-1 PDFs: [WindowsPath('d:/All_Folder/data project/new_papers/Retrieval-augmented generation for continuous anomaly alerts/Dataset/RAG corpus (medical literatureguidelines)/2017 ACCAHAHRS — Evaluation of Patients with Syncope.pdf'), WindowsPath('d:/All_Folder/data project/new_papers/Retrieval-augmented generation for continuous anomaly alerts/Dataset/RAG corpus (medical literatureguidelines)/2017 AHA_ACC_HRS — Management of Ventricular Arrhythmias.pdf'), WindowsPath('d:/All_Folder/data project/new_papers/Retrieval-augmented generation for continuous anomaly alerts/Dataset/RAG corpus (medical literatureguidelines)/ESC 2018 — Guidelines for the Diagnosis and Management of Syncope.pdf'), WindowsPath('d:/All_Folder/data project/new_papers/Retrieval-augmented generation for continuous anomaly alerts/Dataset/RAG corpus (medical literatureguidelines)/ESC 2022 — Guidelines for Ventricular Arrhythmias & SCD.pdf')]
Tier-2 buckets: ['01_ppg_arrhythmia', '02_ppg_signal_quality', '03_ecg_anomaly

 # === Load Tier-1 Corpus ===

 ### === PDF loader ===

In [37]:
def load_tier1():
    """Load all 4 Tier-1 guideline PDFs. Returns list of {source, text}."""
    docs = []
    for pdf_path in sorted(TIER1_DIR.glob("*.pdf")):
        doc = fitz.open(pdf_path)
        text = "\n".join(page.get_text() for page in doc)
        doc.close()
        docs.append({
            "source": pdf_path.stem,   # filename without extension
            "tier": "tier1",
            "text": text,
        })
        print(f"  {pdf_path.stem}: {len(text):,} chars")
    return docs
tier1_docs = load_tier1()
print(f"\nTier-1 loaded: {len(tier1_docs)} documents")
print(f"Total chars: {sum(len(d['text']) for d in tier1_docs):,}")

  2017 ACCAHAHRS — Evaluation of Patients with Syncope: 361,430 chars
  2017 AHA_ACC_HRS — Management of Ventricular Arrhythmias: 687,839 chars
  ESC 2018 — Guidelines for the Diagnosis and Management of Syncope: 359,141 chars
  ESC 2022 — Guidelines for Ventricular Arrhythmias & SCD: 725,258 chars

Tier-1 loaded: 4 documents
Total chars: 2,133,668


# === Load Tier-2 Corpus ===

### === Markdown loader ===

In [38]:
def load_tier2():
    """Load all Tier-2 markdown articles. Returns list of {source, bucket, text}."""
    docs = []
    for bucket_dir in sorted(TIER2_DIR.iterdir()):
        if not bucket_dir.is_dir():
            continue
        md_files = sorted(bucket_dir.glob("*.md"))
        for md_path in md_files:
            text = md_path.read_text(encoding="utf-8")
            docs.append({
                "source": md_path.stem,
                "bucket": bucket_dir.name,
                "tier": "tier2",
                "text": text,
            })
        print(f"  {bucket_dir.name}: {len(md_files)} articles")
    return docs
tier2_docs = load_tier2()
print(f"\nTier-2 loaded: {len(tier2_docs)} articles")
print(f"Total chars: {sum(len(d['text']) for d in tier2_docs):,}")

  01_ppg_arrhythmia: 50 articles
  02_ppg_signal_quality: 25 articles
  03_ecg_anomaly_ml: 45 articles
  04_wearable_stress: 35 articles
  05_continuous_monitoring: 25 articles
  06_biosignal_methods: 20 articles

Tier-2 loaded: 200 articles
Total chars: 12,918,008


# === Chunk Corpus ===

### === Chunking function ===

In [39]:
def chunk_text(text, chunk_words=CHUNK_WORDS, overlap=OVERLAP_WORDS):
    """Split text into overlapping word chunks."""
    words = text.split()
    if len(words) <= chunk_words:
        return [text]
    
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_words
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_words - overlap   # step forward with overlap
    return chunks
def chunk_corpus(docs, doc_type):
    """Chunk all docs; return list of {text, source, tier, bucket, chunk_idx}."""
    all_chunks = []
    for doc in docs:
        chunks = chunk_text(doc["text"])
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "text": chunk,
                "source": doc["source"],
                "tier": doc["tier"],
                "bucket": doc.get("bucket", ""),
                "chunk_idx": i,
            })
    print(f"{doc_type}: {len(docs)} docs → {len(all_chunks)} chunks")
    return all_chunks
tier1_chunks = chunk_corpus(tier1_docs, "Tier-1")
tier2_chunks = chunk_corpus(tier2_docs, "Tier-2")
all_chunks = tier1_chunks + tier2_chunks
print(f"\nTotal chunks to embed: {len(all_chunks):,}")
print(f"Avg chunk length: {np.mean([len(c['text'].split()) for c in all_chunks]):.0f} words")

Tier-1: 4 docs → 679 chunks
Tier-2: 200 docs → 4053 chunks

Total chunks to embed: 4,732
Avg chunk length: 489 words


# === Embed + Index (ChromaDB) ===

In [40]:
# Load embedding model
print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)
print(f"Loaded: {EMBED_MODEL} (dim={embedder.get_sentence_embedding_dimension()})")
# Connect to ChromaDB (persistent)
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
# Create/get collection
COLLECTION_NAME = "medical_corpus"
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"description": "Tier-1 guidelines + Tier-2 literature"}
)
print(f"Collection: {COLLECTION_NAME}")
print(f"Current size: {collection.count()}")
# Only embed if collection is empty (avoids re-embedding on re-run)
if collection.count() == 0:
    print(f"\nEmbedding {len(all_chunks):,} chunks...")
    t0 = time.time()
    
    # Batch embed for speed
    texts = [c["text"] for c in all_chunks]
    metadatas = [{k: v for k, v in c.items() if k != "text"} for c in all_chunks]
    ids = [f"chunk_{i}" for i in range(len(all_chunks))]
    
    # Embed in batches of 64
    BATCH = 64
    all_embeddings = []
    for i in tqdm(range(0, len(texts), BATCH)):
        batch = texts[i:i+BATCH]
        embs = embedder.encode(batch, show_progress_bar=False)
        all_embeddings.extend(embs)
    
    # Add to collection
    collection.add(
        embeddings=all_embeddings,
        documents=texts,
        metadatas=metadatas,
        ids=ids
    )
    elapsed = time.time() - t0
    print(f"\nDone! {len(all_chunks):,} chunks embedded in {elapsed:.1f}s")
    print(f"Final collection size: {collection.count()}")
else:
    print(f"\nCollection already populated ({collection.count()} chunks). Skipping embed.")

Loading embedding model...
Loaded: all-MiniLM-L6-v2 (dim=384)
Collection: medical_corpus
Current size: 4732

Collection already populated (4732 chunks). Skipping embed.


 # === Sanity Retrieval Test ===

In [41]:
def retrieve(query, top_k=TOP_K, pool=20, max_per_source=1):
    """Embed query and retrieve a DIVERSE top-k from ChromaDB.
    Pulls a `pool`-chunk candidate list, then greedily picks at most
    `max_per_source` chunks per source document. Without this, template-style
    queries let 1-2 dominant papers fill every context slot."""
    query_emb = embedder.encode([query])
    results = collection.query(
        query_embeddings=query_emb.tolist(),
        n_results=pool,
        include=["documents", "metadatas", "distances"]
    )
    docs  = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0]
    picked, counts = [], {}
    for i, m in enumerate(metas):
        c = counts.get(m["source"], 0)
        if c >= max_per_source:
            continue
        picked.append(i)
        counts[m["source"]] = c + 1
        if len(picked) == top_k:
            break
    if len(picked) < top_k:   # fallback: fill leftover slots
        for i in range(len(docs)):
            if i not in picked:
                picked.append(i)
                if len(picked) == top_k:
                    break
    return {
        "documents": [[docs[i] for i in picked]],
        "metadatas":  [[metas[i] for i in picked]],
        "distances":  [[dists[i] for i in picked]],
    }
def show_results(query, top_k=3):
    """Pretty-print retrieval results for a query."""
    print(f"\n{'='*70}")
    print(f"QUERY: {query}")
    print('='*70)
    results = retrieve(query, top_k=top_k)
    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    )):
        print(f"\n--- Result {i+1} (distance={dist:.3f}) ---")
        print(f"Source: {meta['source']}")
        print(f"Tier: {meta['tier']}  |  Bucket: {meta.get('bucket', 'N/A')}")
        print(f"Text: {doc[:300]}...")
# 5 test queries spanning different topics
show_results("How does PPG detect atrial fibrillation?")
show_results("What are the causes of false alarms in wearable heart rate monitors?")
show_results("ESC guideline for management of ventricular arrhythmias")
show_results("How is stress detected from electrodermal activity?")
show_results("Isolation Forest algorithm for biosignal anomaly detection")


QUERY: How does PPG detect atrial fibrillation?

--- Result 1 (distance=0.702) ---
Source: PMC12635274_fibricheck_detection_capabilities_for_atrial_fibrillation_fd
Tier: tier2  |  Bucket: 01_ppg_arrhythmia
Text: # Fibricheck detection capabilities for atrial fibrillation (FDA-AF): a multicenter validation study **Authors:** Sollee J, Cheema B, Slotwiner D, Volodarskiy A, Desteghe L, Buyck C, Heidbuchel H, Stavrakis S, Pison L, Nuyens D, Rivero-Ayerza M, Van Herendael H, Thomas J **Journal:** NPJ Digit Med *...

--- Result 2 (distance=0.712) ---
Source: PMC12925684_diagnostic_performance_of_two_commercially_available_ppgbase
Tier: tier2  |  Bucket: 01_ppg_arrhythmia
Text: # Diagnostic performance of two commercially available, PPG-based smartphone applications to detect atrial fibrillation **Authors:** Szonyi MD, Gausz FD, Bocz B, Pilecky D, Muk B, Banfi-Bacsardi F, Foldesi C, Andreka P, Szili-Torok T, Kupo P, Vamos M **Journal:** Eur Heart J Digit Health **DOI:** 10...

--- Result 3 (

# === Load Flagged Windows + Build Queries ===

### === Load the handoff file ===

In [42]:
df = pd.read_parquet(FLAGGED_PATH)
print(f"Loaded {len(df)} flagged windows")
print(df[["subject","t_start_sec","t_end_sec","score_if","score_lof"]].head())

Loaded 398 flagged windows
   subject  t_start_sec  t_end_sec  score_if  score_lof
0        1           30         60    0.4335     1.3521
1        1          180        210    0.4375     1.3580
2        1          390        420    0.4250     1.2885
3        1         2280       2310    0.4428     1.3334
4        1         2940       2970    0.5264     1.5333


### === Query builder ===

In [ ]:
def build_query(row):
    """Turn a flagged window's features into a natural-language RAG query.

    Deviation-aware: leads with the 2 channels that deviate most (z-scored
    across all flagged windows) and injects topic keywords for them, so
    semantically different anomalies retrieve from different corpus
    neighborhoods. (Identical template queries collapsed retrieval to ~11
    documents corpus-wide; this raises it to ~50.)"""
    parts = []

    # 1. Anomaly character
    if row["flag_if"] and row["flag_lof"]:
        parts.append("Robust anomaly confirmed by two independent detectors.")
    elif row["flag_if"]:
        parts.append("Subtle distributed anomaly, gradual drift from baseline.")
    else:
        parts.append("Abrupt isolated spike in wearable biosignals.")

    # 2. Top-2 deviating channels (z-scores precomputed below)
    dev = sorted(CHANNELS_Z, key=lambda c: -abs(CHANNELS_Z[c][row.name]))[:2]
    for c in dev:
        direction = "elevated" if CHANNELS_Z[c][row.name] > 0 else "reduced"
        parts.append(f"{direction} {c} (z={CHANNELS_Z[c][row.name]:+.1f}, mean={row[f'{c}__mean']:.2f})")

    # 3. Topic keywords for the deviating channels -> steers retrieval
    parts.append("Relevant topics: " + " ".join(TOPIC_PHRASES[c] for c in dev) + ".")

    # 4. Remaining channel readings for context
    others = ", ".join(f"{c} mean={row[f'{c}__mean']:.2f}" for c in CHANNELS_Z if c not in dev)
    parts.append(f"Other readings: {others}.")

    return " ".join(parts)

TOPIC_PHRASES = {
    "ecg":       "electrocardiogram rhythm irregularity heart rate variability arrhythmia ectopic beats",
    "resp":      "respiration rate breathing pattern tachypnea bradypnea ventilation",
    "bvp":       "photoplethysmography pulse waveform amplitude perfusion signal quality motion artifact",
    "wrist_eda": "electrodermal activity skin conductance sympathetic stress arousal sweat response",
    "wrist_temp":"skin temperature thermal perfusion vasomotor ambient temperature sensor effects",
}

# z-scores of each channel mean across all flagged windows (relative deviation)
CHANNELS_Z = {
    c: (df[f"{c}__mean"] - df[f"{c}__mean"].mean()) / df[f"{c}__mean"].std()
    for c in ["ecg", "resp", "bvp", "wrist_eda", "wrist_temp"]
}

# Test on first 3 windows
for i in range(3):
    q = build_query(df.iloc[i])
    print(f"\nWindow {i}: subject={df.iloc[i]['subject']}, t={df.iloc[i]['t_start_sec']}s")
    print(f"  Query: {q}")


# === Grounded Explanation Generator ===

### === System prompt + generation function ===

In [ ]:
SYSTEM_PROMPT = """You are a clinical decision-support assistant that explains wearable biosignal anomalies.
You receive:
1. A description of an anomaly detected in a 30-second window of wearable signals.
2. Retrieved excerpts from peer-reviewed clinical guidelines and research articles.
STRICT RULES (never violate):
- Answer ONLY using the provided retrieved context.
- Cite the source document for every clinical claim. Format: [Source Name].
- If the retrieved context does not cover the anomaly, say: "The retrieved context is insufficient to explain this pattern."
- NEVER invent facts, numbers, citations, or medical conclusions not present in the context.
- This is a research tool, NOT a diagnostic device. State this once at the end.
- Keep the explanation under 150 words. Use plain language a nurse could understand.
Output format:
DETECTED: [one-sentence summary of what the anomaly pattern suggests]
EVIDENCE: [what the guidelines/literature say, with citations]
RECOMMENDATION: [what clinical follow-up the guidelines suggest, or "context insufficient"]
DISCLAIMER: Research decision-support tool. Not a diagnostic device. Does not replace clinical judgment."""
import difflib
def canonicalize_citations(text, sources):
    """Canonicalize [PMC...] citations against the retrieved sources.
    1. valid ID (or full source name)  -> kept as short ID
    2. near-miss ID (digit transposition, e.g. PMC12313813 written for
       retrieved PMC13213813)         -> snapped to closest retrieved ID
    3. unresolvable citation           -> dropped entirely
    This fixes LLM digit-swap hallucinations that plain regex cannot."""
    valid = {s.split("_")[0] for s in sources}
    def _fix(m):
        cid = m.group(1)
        if cid in valid:
            return f"[{cid}]"
        close = difflib.get_close_matches(cid, sorted(valid), n=1, cutoff=0.75)
        return f"[{close[0]}]" if close else ""
    return re.sub(r"\[(PMC\d+)[^\]]*\]", _fix, text)

def generate_explanation(query, top_k=TOP_K):
    """Full RAG pipeline: query → retrieve → grounded LLM explanation."""
    t0 = time.time()
    
    # 1. Retrieve
    results = retrieve(query, top_k=top_k)
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    
    # 2. Build context string with source labels
    context_parts = []
    for doc, meta in zip(docs, metas):
        context_parts.append(f"[{meta['source']}]\n{doc}")
    context = "\n\n---\n\n".join(context_parts)
    
    # 3. Call local LLM (think=False disables Qwen3 thinking mode)
    user_msg = f"ANOMALY:\n{query}\n\nRETRIEVED CONTEXT:\n{context}"
    
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        think=False,
        options={
            "temperature": 0.1,
            "num_predict": 500,
            "num_ctx": 10000,
            "num_gpu": 99,
        }
    )
    
    explanation = response["message"]["content"].strip()
    explanation = canonicalize_citations(explanation, [m["source"] for m in metas])
    elapsed = time.time() - t0
    
    return {
        "explanation": explanation,
        "sources": [m["source"] for m in metas],
        "latency_sec": round(elapsed, 1),
    }

### === Test on 3 flagged windows ===

In [45]:
for i in range(3):
    row = df.iloc[i]
    query = build_query(row)
    result = generate_explanation(query)
    
    print(f"\n{'='*70}")
    print(f"WINDOW {i} | Subject {row['subject']} | t={row['t_start_sec']}s | {result['latency_sec']}s")
    print('='*70)
    print(f"QUERY: {query}")
    print(f"\nEXPLANATION:\n{result['explanation']}")
    print(f"\nSOURCES: {result['sources']}")


WINDOW 0 | Subject 1 | t=30s | 8.4s
QUERY: Sharp anomaly spike detected by LOF, respiration rate proxy (mean=0.16), PPG signal amplitude (mean=0.73), wrist EDA=6.66, wrist temperature=32.02

EXPLANATION:
DETECTED: The sharp spike in PPG amplitude and respiration proxy suggests a transient signal artifact or physiological event rather than stable baseline noise.

EVIDENCE: Multiple factors can affect PPG intensity and waveform, including motion artifacts, ambient light, applied pressure, individual variations (skin tone, age), local body temperature, and venous pulsations [PMC13094953]. Signal quality is also highly dependent on posture; for instance, standing with the arm hanging results in lower signal-to-noise ratios compared to supine or sitting positions where the arm is raised to heart height [PMC12204546].

RECOMMENDATION: Verify if the patient was moving (motion artifact) or had their wrist position changed immediately before detection. Check for external light interference or 

# === Batch Generate Explanations ===

In [46]:
results_all = []
for i in tqdm(range(len(df)), desc="Generating explanations"):
    row = df.iloc[i]
    query = build_query(row)
    result = generate_explanation(query)
    
    results_all.append({
        "subject": row["subject"],
        "t_start_sec": row["t_start_sec"],
        "t_end_sec": row["t_end_sec"],
        "score_if": row["score_if"],
        "score_lof": row["score_lof"],
        "query": query,
        "explanation": result["explanation"],
        "sources": result["sources"],
        "latency_sec": result["latency_sec"],
    })
# Save to CSV
df_results = pd.DataFrame(results_all)
df_results.to_csv(OUTPUT_DIR / "rag_explanations.csv", index=False)
print(f"Generated {len(df_results)} explanations")
print(f"Avg latency: {df_results['latency_sec'].mean():.1f}s")
print(f"Total time: {df_results['latency_sec'].sum() / 60:.1f} min")

Generating explanations: 100%|██████████| 398/398 [1:04:27<00:00,  9.72s/it]


Generated 398 explanations
Avg latency: 9.7s
Total time: 64.5 min


# === Evaluation Framework ===

# === RE-ENTRY POINT (fresh kernel) ===

To reproduce the evaluation without re-running generation:

1. Run the **imports** and **paths** cells at the top.
2. Run every cell from here down, **skipping** the LLM cells
   (generation test, batch generation, llama sample-judge, judge-all).
   Their results are already saved in rag_evaluation_v2.csv.
3. The final API-judge cell makes **zero** API calls (all cached in
   api_judge_checkpoint.jsonl) and prints the agreement report.

**Never Run All** — batch generation would regenerate explanations,
invalidating the API checkpoint (~398 paid re-calls).


### === Load and inspect generated explanations ===

In [4]:
df_results = pd.read_csv(OUTPUT_DIR / "rag_explanations.csv")
print(f"Loaded {len(df_results)} explanations")
# Show a sample to refresh what we're evaluating
sample = df_results.iloc[0]
print(f"\n--- Sample Explanation ---")
print(f"Query: {sample['query']}")
print(f"\nExplanation:\n{sample['explanation']}")
print(f"\nSources: {sample['sources']}")

Loaded 398 explanations

--- Sample Explanation ---
Query: Abrupt isolated spike in wearable biosignals. reduced wrist_eda (z=-0.5, mean=6.66) reduced wrist_temp (z=-0.5, mean=32.02) Relevant topics: electrodermal activity skin conductance sympathetic stress arousal sweat response skin temperature thermal perfusion vasomotor ambient temperature sensor effects. Other readings: ecg mean=0.00, resp mean=0.16, bvp mean=0.73.

Explanation:
DETECTED: The abrupt spike combined with reduced skin temperature and EDA likely represents a motion artifact or sensor displacement rather than genuine stress, as wrist-worn devices are prone to signal inaccuracies during movement [PMC12635167].
EVIDENCE: Literature notes that collecting EDA signals using wristbands is not very accurate and artifacts can affect data collection results; additionally, motion artifacts from electrode displacement produce sharp transients indistinguishable from genuine stress responses [PMC12828444]. Skin temperature also va

### === Parse explanations into components ===

In [5]:
def parse_explanation(text):
    """Split a generated explanation into its structured components."""
    parts = {"detected": "", "evidence": "", "recommendation": "", "disclaimer": ""}
    current = None
    
    for line in text.split("\n"):
        line = line.strip()
        if line.startswith("DETECTED:"):
            current = "detected"
            parts[current] = line[len("DETECTED:"):].strip()
        elif line.startswith("EVIDENCE:"):
            current = "evidence"
            parts[current] = line[len("EVIDENCE:"):].strip()
        elif line.startswith("RECOMMENDATION:"):
            current = "recommendation"
            parts[current] = line[len("RECOMMENDATION:"):].strip()
        elif line.startswith("DISCLAIMER:"):
            current = "disclaimer"
            parts[current] = line[len("DISCLAIMER:"):].strip()
        elif current and line:
            parts[current] += " " + line
    
    return parts
# Apply to all results
df_parsed = df_results.copy()
parsed = df_parsed["explanation"].apply(parse_explanation).apply(pd.Series)
df_parsed = pd.concat([df_parsed, parsed], axis=1)
print("Parsed components.")
print(df_parsed[["detected","evidence"]].head(2))

Parsed components.
                                            detected                                           evidence
0  The abrupt spike combined with reduced skin te...  Literature notes that collecting EDA signals u...
1  The abrupt spike suggests a potential motion a...  Literature notes that collecting EDA using wri...


# ===  Citation Accuracy (Programmatic, 100% objective) ===

In [6]:
import re
def check_citations(explanation, sources):
    """
    Check that every [Citation] in explanation exists in the sources list.
    Returns {n_citations, n_valid, all_valid, invalid_citations}.
    """
    # Extract all [PMC...] or bracketed citations
    cited = re.findall(r'\[([A-Za-z0-9_]+)\]', explanation)
    cited = [c for c in cited if len(c) > 5]  # filter out short brackets
    
    # Extract PMC IDs from sources (strip suffixes)
    valid_ids = set()
    for s in sources:
        # Take first underscore-separated token (the PMC ID)
        pmc = s.split("_")[0] if "_" in s else s
        valid_ids.add(pmc)
        valid_ids.add(s)  # also keep full name
    
    n_valid = sum(1 for c in cited if c in valid_ids or any(c in v for v in valid_ids))
    n_invalid = len(cited) - n_valid
    
    return {
        "n_citations": len(cited),
        "n_valid": n_valid,
        "n_invalid": n_invalid,
        "all_valid": n_invalid == 0,
    }
# Test on sample
sample = df_parsed.iloc[0]
cit = check_citations(sample["evidence"] + " " + sample["recommendation"], 
                       eval(sample["sources"]))
print("Sample citation check:")
print(f"  Total citations: {cit['n_citations']}")
print(f"  Valid: {cit['n_valid']}")
print(f"  Invalid (hallucinated): {cit['n_invalid']}")
print(f"  All valid: {cit['all_valid']}")

Sample citation check:
  Total citations: 2
  Valid: 2
  Invalid (hallucinated): 0
  All valid: True


### === Citation accuracy on all 398 ===

In [7]:
# Apply citation check to all explanations
citation_results = []
for i in range(len(df_parsed)):
    row = df_parsed.iloc[i]
    full_text = f"{row['detected']} {row['evidence']} {row['recommendation']}"
    sources = eval(row["sources"]) if isinstance(row["sources"], str) else row["sources"]
    cit = check_citations(full_text, sources)
    citation_results.append(cit)
cit_df = pd.DataFrame(citation_results)
df_parsed = pd.concat([df_parsed.reset_index(drop=True), cit_df], axis=1)
print("=== Citation Accuracy (All 398 Explanations) ===")
print(f"Total citations checked:    {cit_df['n_citations'].sum()}")
print(f"Valid citations:            {cit_df['n_valid'].sum()}")
print(f"Hallucinated citations:     {cit_df['n_invalid'].sum()}")
print(f"Citation accuracy:          {100 * cit_df['n_valid'].sum() / cit_df['n_citations'].sum():.1f}%")
print(f"\nExplanations with ALL valid citations: {cit_df['all_valid'].sum()}/{len(cit_df)} ({100*cit_df['all_valid'].mean():.1f}%)")
print(f"Avg citations per explanation:         {cit_df['n_citations'].mean():.1f}")

=== Citation Accuracy (All 398 Explanations) ===
Total citations checked:    1359
Valid citations:            1359
Hallucinated citations:     0
Citation accuracy:          100.0%

Explanations with ALL valid citations: 398/398 (100.0%)
Avg citations per explanation:         3.4


### === Citation repair (run once after the fix) ===


In [8]:
# === One-off repair: canonicalize invalid citations in existing explanations ===
# Self-contained: defines its own canonicalizer so it runs even if the
# generator cell (which now canonicalizes at generation time) was not re-run.
import re, difflib

def canonicalize_citations(text, sources):
    """Snap near-miss PMC IDs (digit transpositions) to the retrieved source;
    drop unresolvable citations. Mirrors the generator-cell version."""
    valid = {s.split("_")[0] for s in sources}
    def _fix(m):
        cid = m.group(1)
        if cid in valid:
            return f"[{cid}]"
        close = difflib.get_close_matches(cid, sorted(valid), n=1, cutoff=0.75)
        return f"[{close[0]}]" if close else ""
    return re.sub(r"\[(PMC\d+)[^\]]*\]", _fix, text)
# generate_explanation now canonicalizes at generation time (see that cell).
# This pass fixes rows generated BEFORE that fix: near-miss PMC IDs are snapped
# to the retrieved source, unresolvable brackets are dropped, the CSV is
# re-saved, and the objective citation check is re-run.
n_fixed = 0
for i in range(len(df_results)):
    row = df_results.iloc[i]
    srcs = eval(row["sources"]) if isinstance(row["sources"], str) else row["sources"]
    fixed = canonicalize_citations(row["explanation"], srcs)
    if fixed != row["explanation"]:
        df_results.at[i, "explanation"] = fixed
        n_fixed += 1
df_results.to_csv(OUTPUT_DIR / "rag_explanations.csv", index=False)
print(f"Canonicalized citations in {n_fixed}/{len(df_results)} rows; re-saved rag_explanations.csv")

# rebuild df_parsed from repaired explanations + re-run citation check
df_parsed = df_results.copy()
parsed = df_parsed["explanation"].apply(parse_explanation).apply(pd.Series)
df_parsed = pd.concat([df_parsed, parsed], axis=1)
citation_results = []
for i in range(len(df_parsed)):
    row = df_parsed.iloc[i]
    full_text = f"{row['detected']} {row['evidence']} {row['recommendation']}"
    sources = eval(row["sources"]) if isinstance(row["sources"], str) else row["sources"]
    citation_results.append(check_citations(full_text, sources))
cit_df = pd.DataFrame(citation_results)
df_parsed = pd.concat([df_parsed.reset_index(drop=True), cit_df], axis=1)
print("=== Citation Accuracy AFTER repair ===")
print(f"Total citations checked:    {cit_df['n_citations'].sum()}")
print(f"Hallucinated citations:     {cit_df['n_invalid'].sum()}")
print(f"Citation accuracy:          {100 * cit_df['n_valid'].sum() / max(1, cit_df['n_citations'].sum()):.1f}%")
print(f"Explanations with ALL valid citations: {cit_df['all_valid'].sum()}/{len(cit_df)} ({100*cit_df['all_valid'].mean():.1f}%)")

Canonicalized citations in 0/398 rows; re-saved rag_explanations.csv
=== Citation Accuracy AFTER repair ===
Total citations checked:    1359
Hallucinated citations:     0
Citation accuracy:          100.0%
Explanations with ALL valid citations: 398/398 (100.0%)


### === LLM-as-Judge for semantic scoring ===

In [ ]:
JUDGE_PROMPT = """You are an expert evaluator for clinical AI alert systems.
You will see:
1. QUERY: a description of a detected biosignal anomaly
2. SOURCES: the retrieved medical literature excerpts given to the system
3. EXPLANATION: what the system generated
Score the EXPLANATION on three axes. For each, output a score (1, 2, or 3) and a one-line reason.
FAITHFULNESS (Is every claim grounded in the sources?):
  1 = Contains facts/conclusions NOT present in the sources (hallucination)
  2 = Mostly grounded but some claims are vague or slightly extrapolated
  3 = Every claim is traceable to the sources (fully grounded)
RELEVANCE (Is the explanation about THIS specific anomaly?):
  1 = Not about the detected anomaly pattern
  2 = Partially relevant, discusses general wearable topics
  3 = Directly addresses the specific anomaly features described
COMPLETENESS (Does it cover what clinicians need to know?):
  1 = Missing key aspects (what it means, why, what to do)
  2 = Covers some but not all key aspects
  3 = Addresses detection, cause, and recommended action
Output format (use EXACTLY this format, nothing else):
FAITHFULNESS: [score] - [reason]
RELEVANCE: [score] - [reason]
COMPLETENESS: [score] - [reason]"""
def judge_explanation(query, explanation, sources_text, top_k=3):
    """Use independent LLM judge to score an explanation."""
    
    # FIX: no truncation — the judge must see exactly what the generator saw
    sources_excerpt = sources_text
    
    user_msg = f"""QUERY: {query}
SOURCES:
{sources_excerpt}
EXPLANATION:
{explanation}"""
    response = ollama.chat(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        options={
            "temperature": 0.1,
            "num_predict": 300,
            "num_ctx": 10000,
            "num_gpu": 99,
        }
    )
    
    return response["message"]["content"].strip()
def parse_judge_scores(judge_text):
    """Parse judge output into numeric scores."""
    scores = {"faithfulness": 0, "relevance": 0, "completeness": 0}
    reasons = {"faithfulness": "", "relevance": "", "completeness": ""}
    
    for line in judge_text.split("\n"):
        line = line.strip()
        for axis in scores:
            if line.upper().startswith(axis.upper() + ":"):
                rest = line[len(axis)+1:].strip()
                if rest and rest[0].isdigit():
                    scores[axis] = int(rest[0])
                    reasons[axis] = rest[2:].strip("- ").strip()
    
    return scores, reasons
# Test on sample
sample = df_parsed.iloc[0]
query = sample["query"]
explanation = sample["explanation"]
# Get the actual retrieved text for this window (same as generator: top-5, full chunks)
results = retrieve(query, top_k=TOP_K)
sources_text = "\n\n".join(f"[{m['source']}]\n{d}" 
                            for d, m in zip(results["documents"][0], results["metadatas"][0]))
judge_output = judge_explanation(query, explanation, sources_text)
scores, reasons = parse_judge_scores(judge_output)
print("=== Sample Judge Output ===")
print(judge_output)
print(f"\nParsed scores: {scores}")

### === Quick validation: llama3.1:8b judge on a sample ===


In [ ]:
# Validate the judge on a few rows BEFORE committing to the full 398
N_SAMPLE = 5
llama_sample = []
for i in range(N_SAMPLE):
    row = df_parsed.iloc[i]
    query, explanation = row["query"], row["explanation"]
    results = retrieve(query, top_k=TOP_K)
    sources_text = "\n\n".join(f"[{m['source']}]\n{d}"
                                for d, m in zip(results["documents"][0], results["metadatas"][0]))
    t0 = time.time()
    judge_output = judge_explanation(query, explanation, sources_text)
    dt = time.time() - t0
    scores, reasons = parse_judge_scores(judge_output)
    llama_sample.append({**scores, "latency_sec": dt})
    print(f"Row {i}: {dt:5.1f}s | faith={scores['faithfulness']} rel={scores['relevance']} comp={scores['completeness']}")
s = pd.DataFrame(llama_sample)
print(f"\n=== {JUDGE_MODEL} on {N_SAMPLE} rows (judge sees top-5 full chunks, same as generator) ===")
print(f"Avg latency:  {s['latency_sec'].mean():.1f}s")
print(f"Faithfulness: {s['faithfulness'].mean():.2f} / 3")
print(f"Relevance:    {s['relevance'].mean():.2f} / 3")
print(f"Completeness: {s['completeness'].mean():.2f} / 3")
n_fail = (s[["faithfulness","relevance","completeness"]] == 0).any(axis=1).sum()
print(f"Parse failures (any score=0): {n_fail}/{N_SAMPLE}")
print(f"Projected full 398-row run: ~{s['latency_sec'].mean() * 398 / 3600:.1f} h")


# === Judge all 398 ===

In [ ]:
judge_scores_all = []
judge_reasons_all = []
n_source_mismatch = 0
for i in tqdm(range(len(df_parsed)), desc="Judging explanations"):
    row = df_parsed.iloc[i]
    query = row["query"]
    explanation = row["explanation"]
    
    # Re-retrieve with the SAME settings the generator used (top-5, full chunks)
    # so the judge sees exactly what the generator saw.
    results = retrieve(query, top_k=TOP_K)
    sources_text = "\n\n".join(f"[{m['source']}]\n{d}"
                                for d, m in zip(results["documents"][0], results["metadatas"][0]))
    # Sanity check: re-retrieval must reproduce the sources saved at generation time
    saved_sources = eval(row["sources"]) if isinstance(row["sources"], str) else row["sources"]
    retrieved_sources = [m["source"] for m in results["metadatas"][0]]
    if retrieved_sources != saved_sources:
        n_source_mismatch += 1
    
    try:
        judge_output = judge_explanation(query, explanation, sources_text)
        scores, reasons = parse_judge_scores(judge_output)
    except Exception as e:
        scores = {"faithfulness": 0, "relevance": 0, "completeness": 0}
        reasons = {"faithfulness": f"error: {e}", "relevance": "", "completeness": ""}
    
    judge_scores_all.append(scores)
    judge_reasons_all.append(reasons)
# Save to dataframe
scores_df = pd.DataFrame(judge_scores_all)
reasons_df = pd.DataFrame(judge_reasons_all).add_suffix("_reason")
df_eval = pd.concat([
    df_parsed.reset_index(drop=True),
    scores_df,
    reasons_df
], axis=1)
print(f"Source-set mismatches (re-retrieval vs saved): {n_source_mismatch}/{len(df_parsed)}")
df_eval.to_csv(OUTPUT_DIR / "rag_evaluation_v2.csv", index=False)
print(f"Judged {len(df_eval)} explanations")
print(f"Avg faithfulness:  {scores_df['faithfulness'].mean():.2f} / 3")
print(f"Avg relevance:     {scores_df['relevance'].mean():.2f} / 3")
print(f"Avg completeness:  {scores_df['completeness'].mean():.2f} / 3")

# === Optional: API reference judge (OpenRouter) ===
Second judge (larger hosted model) to cross-check the local llama3.1 judge via Cohen's kappa.

**Safety / cost guarantees:**
- API key read **only** from the `OPENROUTER_API_KEY` env var (or a `.env` file in the project root) — never stored in the notebook or outputs.
- **Idempotent:** every successful judgment is checkpointed to `outputs/api_judge_checkpoint.jsonl` immediately. Re-running this cell or the entire notebook makes **zero** new API calls for already-judged rows.
- Failed rows are not checkpointed and are retried on the next run.
- Checkpoint keys include the explanation hash: if explanations are regenerated, only then are rows re-judged.
- Cell skips gracefully (no calls, no crash) when the key is missing.


In [9]:
# === Optional: API reference judge (OpenRouter) ===
# - Key comes ONLY from the OPENROUTER_API_KEY environment variable (or a
#   local .env file in the project root). Never hardcoded, never in outputs.
# - IDEMPOTENT by design: every successful judgment is appended to a JSONL
#   checkpoint immediately. Re-running this cell or the whole notebook makes
#   ZERO new API calls for rows already in the checkpoint (keyed by
#   subject + time window + model + explanation hash, so regenerated
#   explanations are correctly re-judged).
# - Rows that errored are NOT checkpointed -> automatically retried next run.

import os, json, time, hashlib, requests

API_JUDGE_MODEL = "deepseek/deepseek-v4-flash-0731"
API_CHECKPOINT  = OUTPUT_DIR / "api_judge_checkpoint.jsonl"
API_RESULTS_CSV = OUTPUT_DIR / "rag_evaluation_api.csv"
OPENROUTER_URL  = "https://openrouter.ai/api/v1/chat/completions"

def _load_api_key():
    key = os.environ.get("OPENROUTER_API_KEY", "")
    if not key:
        env_file = PROJECT_ROOT / ".env"
        if env_file.exists():
            for line in env_file.read_text(encoding="utf-8").splitlines():
                if line.strip().startswith("OPENROUTER_API_KEY"):
                    key = line.split("=", 1)[1].strip().strip('"').strip("'")
                    break
    return key

def _row_key(row):
    ident = (f"{row['subject']}|{row['t_start_sec']}|{row['t_end_sec']}"
             f"|{API_JUDGE_MODEL}|{hashlib.md5(row['explanation'].encode('utf-8')).hexdigest()[:10]}")
    return hashlib.md5(ident.encode("utf-8")).hexdigest()

def _load_checkpoint():
    done = {}
    n_bad = 0
    if API_CHECKPOINT.exists():
        for line in API_CHECKPOINT.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                # discard checkpointed rows whose output was truncated
                # (any score=0 means the judge format did not parse -> garbage;
                #  those rows are re-sent automatically)
                if 0 in rec.get("scores", {}).values():
                    n_bad += 1
                    continue
                done[rec["key"]] = rec
            except json.JSONDecodeError:
                continue   # torn write from a crash -> row gets retried
    if n_bad:
        print(f"[checkpoint] dropped {n_bad} truncated/garbage record(s); those rows will be re-judged")
    return done

def _api_chat(user_msg, api_key, max_retries=4):
    # DeepSeek V4 Flash is a hybrid reasoning model: thinking is ON by default
    # and reasoning tokens come OUT of max_tokens. With max_tokens=300 the model
    # spent the whole budget thinking and returned content=null (the
    # "'NoneType' object has no attribute 'strip'" errors). Rubric scoring needs
    # no reasoning -> disable it, and give headroom regardless.
    payload = {
        "model": API_JUDGE_MODEL,
        "messages": [
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user",   "content": user_msg},
        ],
        "temperature": 0.1,          # honored once reasoning is disabled
        "max_tokens": 2000,          # headroom; reasoning off keeps this cheap
        "reasoning": {"enabled": False},
    }
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    last_err = None
    for attempt in range(max_retries):
        resp = requests.post(OPENROUTER_URL, headers=headers, json=payload, timeout=180)
        if resp.status_code in (429, 500, 502, 503):
            last_err = f"HTTP {resp.status_code}"
            time.sleep(5 * (attempt + 1))
            continue
        resp.raise_for_status()
        data = resp.json()
        try:
            msg = data["choices"][0]["message"]
        except (KeyError, IndexError):
            last_err = f"unexpected response shape: {str(data)[:200]}"
            time.sleep(3 * (attempt + 1))
            continue
        content = msg.get("content")
        if content and content.strip():
            return content.strip()
        last_err = f"empty content (finish_reason={data['choices'][0].get('finish_reason')})"
        time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"OpenRouter failed after {max_retries} attempts: {last_err}")

api_key = _load_api_key()
if not api_key:
    print("OPENROUTER_API_KEY not found -> API judge SKIPPED (nothing sent; local judge results stand).")
    print("  PowerShell (this session): $env:OPENROUTER_API_KEY = 'sk-or-...'")
    print("  Persistent:                setx OPENROUTER_API_KEY 'sk-or-...'")
    print("  Or create .env in project root:  OPENROUTER_API_KEY=sk-or-...")
else:
    done = _load_checkpoint()
    n_skip = len(done)
    n_new, n_err = 0, 0
    with open(API_CHECKPOINT, "a", encoding="utf-8") as fchk:
        for i in tqdm(range(len(df_parsed)), desc=f"API judge ({API_JUDGE_MODEL})"):
            row = df_parsed.iloc[i]
            key = _row_key(row)
            if key in done:
                continue                       # already judged -> NO API call
            results = retrieve(row["query"], top_k=TOP_K)
            sources_text = "\n\n".join(f"[{m['source']}]\n{d}"
                                       for d, m in zip(results["documents"][0], results["metadatas"][0]))
            user_msg = f"QUERY: {row['query']}\nSOURCES:\n{sources_text}\nEXPLANATION:\n{row['explanation']}"
            try:
                raw = _api_chat(user_msg, api_key)
                scores, reasons = parse_judge_scores(raw)
                if 0 in scores.values():
                    # incomplete/truncated judge output: do NOT checkpoint,
                    # raise so the row is retried on the next run
                    raise RuntimeError("incomplete judge output (score=0 parsed)")
                rec = {"key": key, "i": i, "model": API_JUDGE_MODEL,
                       "subject": int(row["subject"]), "t_start_sec": int(row["t_start_sec"]),
                       "scores": scores, "reasons": reasons, "raw": raw,
                       "ts": time.strftime("%Y-%m-%d %H:%M:%S")}
                fchk.write(json.dumps(rec, ensure_ascii=False) + "\n")
                fchk.flush()                   # crash-safe: survive kernel death
                done[key] = rec
                n_new += 1
            except Exception as e:
                n_err += 1                     # NOT checkpointed -> retried next run
                print(f"\nrow {i} error (will retry on next run): {e}")
            time.sleep(0.5)                    # gentle on rate limits
    print(f"\nAPI calls this run: {n_new} | skipped from checkpoint: {n_skip} | errors: {n_err}")

# --- Assemble API-judge CSV from the checkpoint (makes NO API calls) ---
done = _load_checkpoint()
recs = sorted(done.values(), key=lambda r: r["i"])
if recs:
    api_cols = pd.DataFrame(
        [{"i": r["i"],
          "api_faithfulness": r["scores"]["faithfulness"],
          "api_relevance":    r["scores"]["relevance"],
          "api_completeness": r["scores"]["completeness"]} for r in recs]
    )
    # a re-judged row (changed explanation) has >1 record for the same i:
    # keep the LATEST (file order = chronological) one per row index
    api_cols = (api_cols.sort_values("i", kind="stable")
                          .drop_duplicates(subset="i", keep="last")
                          .set_index("i"))
    # Base = local-judge results. Prefer in-memory df_eval; otherwise load the
    # saved local-judge CSV (avoids re-running the 30-min local judge AND
    # prevents overwriting the good CSV with a local-scores-free copy).
    if "df_eval" in globals():
        base = df_eval
    elif (OUTPUT_DIR / "rag_evaluation_v2.csv").exists():
        base = pd.read_csv(OUTPUT_DIR / "rag_evaluation_v2.csv")
        print("[base] loaded local-judge scores from rag_evaluation_v2.csv")
    else:
        base = df_parsed
        print("[base] WARNING: no local-judge scores found; agreement block will be skipped")
    df_api = base.copy()
    for col in api_cols.columns:
        df_api[col] = api_cols[col]
    df_api.to_csv(API_RESULTS_CSV, index=False)
    print(f"Saved {API_RESULTS_CSV.name} ({len(api_cols)}/{len(df_parsed)} rows judged so far)")
    if len(api_cols) == len(df_parsed) and {"faithfulness", "relevance", "completeness"}.issubset(df_api.columns):
        from sklearn.metrics import cohen_kappa_score
        import numpy as np
        print("\n=== Judge agreement: local llama3.1:8b vs API reference judge ===")
        print(f"{'axis':14s} {'raw agree':>9s} {'within-1':>9s} {'kappa':>6s}  note")
        for axis in ["faithfulness", "relevance", "completeness"]:
            a, b = df_api[f"api_{axis}"], df_api[axis]
            mask = (a > 0) & (b > 0)
            a_m, b_m = a[mask], b[mask]
            agree  = (a_m == b_m).mean()
            within = ((a_m - b_m).abs() <= 1).mean()
            try:
                kappa = cohen_kappa_score(a_m, b_m)
            except Exception:
                kappa = float("nan")
            # constant-rater detection: kappa is meaningless when either judge
            # assigns the same score to every row (observed == chance agreement)
            note = ""
            if a_m.nunique() == 1 or b_m.nunique() == 1:
                const_judge = "API" if a_m.nunique() == 1 else "local"
                note = f"kappa undefined ({const_judge} judge is constant)"
            print(f"{axis:14s} {agree:9.1%} {within:9.1%} {kappa:6.2f}  {note}")
        # headline safety claim, cross-validated by two model families
        n_hallu = int((df_api["api_faithfulness"] == 1).sum())
        n_any1  = int((df_api[["api_faithfulness", "api_relevance", "api_completeness"]] == 1).any(axis=1).sum())
        print(f"\nHallucination flags (score=1): {n_hallu}/398 faithfulness, {n_any1}/398 on any axis")
        print("API judge means: " + ", ".join(
            f"{k.replace('api_','')}={df_api[k].mean():.2f}"
            for k in ["api_faithfulness", "api_relevance", "api_completeness"]))


[checkpoint] dropped 20 truncated/garbage record(s); those rows will be re-judged

API calls this run: 0 | skipped from checkpoint: 398 | errors: 0
[checkpoint] dropped 20 truncated/garbage record(s); those rows will be re-judged
[base] loaded local-judge scores from rag_evaluation_v2.csv
Saved rag_evaluation_api.csv (398/398 rows judged so far)

=== Judge agreement: local llama3.1:8b vs API reference judge ===
axis           raw agree  within-1  kappa  note
faithfulness       59.2%    100.0%   0.00  kappa undefined (local judge is constant)
relevance          79.8%    100.0%   0.00  kappa undefined (local judge is constant)
completeness       52.4%    100.0%  -0.01  

Hallucination flags (score=1): 0/398 faithfulness, 0/398 on any axis
API judge means: faithfulness=2.59, relevance=2.80, completeness=2.47
